# Eventos, estado y cálculo

[00_indice](00_indice.ipynb) · [01_python_y_flet](01_python_y_flet.ipynb) · [02_modelos](02_modelos.ipynb) · [03_controles](03_controles.ipynb) · [04_graficas](04_graficas.ipynb) · [05_eventos](05_eventos.ipynb) · [06_referencia](06_referencia.ipynb) · [07_validacion_e_integracion](07_validacion_e_integracion.ipynb) · [08_conectar_algoritmo](08_conectar_algoritmo.ipynb) · [09_demo_engine_paso_a_paso](09_demo_engine_paso_a_paso.ipynb)

**AM / Evolución Lab · Flet 1.0.4 · Python 3**

## Estado
page aloja controles; engine implementa el motor; config es la última configuración aplicada; snapshots son Generation; history contiene mínimos; index es generación visible; total iteraciones; playing reproducción; busy cálculo en curso; revision token de invalidación; is_demo controla etiqueta de demostración.

Controles: function, dimensions, size, generations, selection, mutation_label, mutation, message, formula, editor, metrics, function_chart, history_chart, table, timeline, play_button, progress, apply_button, edit_button, action_hint, actions, result_summary, menu, configuration_view y results_view. router mantiene navegación e historial. body es la columna desplazable; raíz no tiene scroll, por lo que menú y barra actions permanecen visibles. _on_route_change alterna Ver resultados y Editar configuración.

Al navegar se programa _scroll_to_start mediante page.run_task: espera un turno con asyncio.sleep(0) y llama await body.scroll_to(offset=0,duration=0). Así los resultados se abren desde arriba y la gráfica no queda oculta por el desplazamiento del formulario anterior. Sólo cambia desplazamiento, conservando datos y ediciones.

## Preparación
_prepare_population crea muestra inicial sin ejecutar el motor. build organiza vistas. show_view(name) delega navegación al router; _on_route_change(name) pausa y sincroniza menú. change_objective reinicia esquema y vacía lista al cambiar dominio/dimensiones. configure genera población sin aplicarla. population_changed(individuals) marca cambios; la cantidad aplicada deriva de la lista, no del campo del generador.

invalidate detiene reproducción y marca cambios pendientes. change_mutation actualiza etiqueta; settings devuelve AlgorithmConfig con mutation.value/100. Si hay campos pendientes o edición activa, settings pide guardar o cancelar antes de aplicar.

## Ejecutar motor
apply es async: valida entradas, pausa, marca busy, deshabilita botón, captura token, ejecuta `await asyncio.to_thread(self.engine.run, config)` y valida generaciones. El hilo permite atender eventos mientras trabaja el motor. La configuración es inmutable para evitar modificar sus vectores durante el cálculo.

Si cambió revision, el resultado se descarta y la configuración aplicada anterior se conserva. Esta invalidación no cancela físicamente el hilo del motor; una cancelación real requiere cooperación del motor. finally restablece botón y busy incluso cuando hay errores.

No necesitas cambiar render para integrar operadores: render presenta una Generation ya calculada, métricas, mapa/curva, lista, historial y progreso. La minimización usa min(scores); si tu algoritmo maximiza, adapta ese criterio en render/history y documentación.

## Recorrido
seek lee slider y pausa; step(delta) limita índice; reset vuelve a 0. toggle_play usa await asyncio.sleep(0.35); token controla si otra interacción canceló el recorrido. Al terminar restaura Reproducir. _pause incrementa revision y detiene reproducciones antiguas.

e=None permite callbacks y llamadas internas; self es instancia. lambda e,i=index captura cada fila. Los cambios de datos llaman page.update después de agrupar propiedades.
